# Sprint 6 – Notebook 12 – Feature Leakage

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)
n = 120
customers = pd.DataFrame({
    'Customer_ID': np.arange(1001, 1001+n),
    'Age': rng.integers(19, 66, n),
    'Annual_Income': rng.integers(250000, 1500000, n),
    'Quantity': rng.integers(1, 7, n),
    'Product_Price': rng.integers(200, 5000, n),
    'City': rng.choice(['Bengaluru','Mysuru','Hassan'], n),
    'Membership_Type': rng.choice(['Basic','Silver','Gold'], n),
    'Purchase_Date': pd.date_range('2025-01-01', periods=n, freq='D'),
})
customers['Total_Spending'] = customers['Quantity'] * customers['Product_Price']
customers['Repeat_Purchase'] = ((customers['Total_Spending'] > 8500) | (customers['Membership_Type'] == 'Gold')).astype(int)
customers.head()

,Customer_ID,Age,Annual_Income,Quantity,Product_Price,City,Membership_Type,Purchase_Date,Total_Spending,Repeat_Purchase
0,1001,23,799516,1,4259,Bengaluru,Silver,2025-01-01,4259,0
1,1002,55,1085503,4,4475,Hassan,Gold,2025-01-02,17900,1
2,1003,49,1068036,1,1272,Bengaluru,Gold,2025-01-03,1272,1
3,1004,39,838870,4,4488,Hassan,Basic,2025-01-04,17952,1
4,1005,39,1318850,4,756,Mysuru,Gold,2025-01-05,3024,1


## What is Feature Leakage?

### Definition
What is Feature Leakage? is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply what is feature leakage? while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, what is feature leakage? can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [3]:
bad = customers[['Age','Total_Spending','Repeat_Purchase']].copy();bad['Copied_Target']=bad['Repeat_Purchase'];print(bad.head(3))

   Age  Total_Spending  Repeat_Purchase  Copied_Target
0   23            4259                0              0
1   55           17900                1              1
2   49            1272                1              1


### Explanation
Copied_Target is exactly the answer the model is supposed to predict. Such a column makes evaluation meaningless because it reveals the outcome. Remove it before training. In practice, verify when every feature becomes available.

## Target Leakage

### Definition
Target Leakage is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply target leakage while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, target leakage can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [5]:
print('Incorrect: use Repeat_Purchase as an input');print('Correct inputs:', ['Age','Annual_Income','Quantity','Product_Price'])

Incorrect: use Repeat_Purchase as an input
Correct inputs: ['Age', 'Annual_Income', 'Quantity', 'Product_Price']


### Explanation
This example demonstrates target leakage with the customer shopping dataset. Run the cell to inspect its result. Notice which columns are used and what changes in the output. In a real project, compare the result with the original data and check that no future or target information has leaked.

## Train-Test Leakage

### Definition
Train-Test Leakage is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply train-test leakage while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, train-test leakage can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [7]:
X = customers[['Age','Annual_Income']];tr,te=train_test_split(X,test_size=.25,random_state=42);scaler=StandardScaler().fit(tr);print('Train mean:',scaler.mean_.round(1));print('Test transformed shape:',scaler.transform(te).shape)

Train mean: [4.230000e+01 8.964736e+05]
Test transformed shape: (30, 2)


### Explanation
This example demonstrates train-test leakage with the customer shopping dataset. Run the cell to inspect its result. Notice which columns are used and what changes in the output. In a real project, compare the result with the original data and check that no future or target information has leaked.

## Temporal Leakage

### Definition
Temporal Leakage is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply temporal leakage while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, temporal leakage can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [9]:
history=customers.sort_values('Purchase_Date').copy();history['Previous_Spending']=history['Total_Spending'].shift(1);print(history[['Purchase_Date','Total_Spending','Previous_Spending']].head())

  Purchase_Date  Total_Spending  Previous_Spending
0    2025-01-01            4259                NaN
1    2025-01-02           17900             4259.0
2    2025-01-03            1272            17900.0
3    2025-01-04           17952             1272.0
4    2025-01-05            3024            17952.0


### Explanation
A historical feature should use only events occurring before the prediction time. The shifted value uses the previous transaction rather than a future one. If predicting per customer, perform the shift separately within each customer history. A random split is not appropriate for all time-dependent tasks.

## Post-Outcome Features

### Definition
Post-Outcome Features is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply post-outcome features while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, post-outcome features can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [11]:
example=customers[['Age','Repeat_Purchase']].head(3).copy();example['Post_Outcome_Reward']=example['Repeat_Purchase']*100;print(example)

   Age  Repeat_Purchase  Post_Outcome_Reward
0   23                0                    0
1   55                1                  100
2   49                1                  100


### Explanation
Rewards issued after repeat purchase are unavailable when predicting that purchase. Their strong relationship with the target is misleading. Features must exist before the decision point. Drop post-outcome columns from the model inputs.

## Leakage Through Aggregations

### Definition
Leakage Through Aggregations is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply leakage through aggregations while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, leakage through aggregations can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [13]:
events=pd.DataFrame({'Customer_ID':[1,1,1,2,2],'Day':[1,2,3,1,2],'Amount':[100,200,400,50,150]});events['Wrong_All_Time_Total']=events.groupby('Customer_ID')['Amount'].transform('sum');events['Correct_Past_Total']=events.groupby('Customer_ID')['Amount'].transform(lambda s:s.shift().fillna(0).cumsum());print(events)

   Customer_ID  Day  Amount  Wrong_All_Time_Total  Correct_Past_Total
0            1    1     100                   700                 0.0
1            1    2     200                   700               100.0
2            1    3     400                   700               300.0
3            2    1      50                   200                 0.0
4            2    2     150                   200                50.0


### Explanation
The all-time total includes future purchases for early transactions. Past_Total uses only earlier events for each customer. Whether the current transaction can be included depends on the prediction timestamp. Define the prediction moment before creating aggregate features.

## Leakage Through Target Encoding

### Definition
Leakage Through Target Encoding is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply leakage through target encoding while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, leakage through target encoding can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [15]:
tiny=pd.DataFrame({'City':['A','A','B','B'],'Target':[1,0,1,0]});tiny['Wrong_Target_Mean']=tiny.groupby('City')['Target'].transform('mean');print(tiny)

  City  Target  Wrong_Target_Mean
0    A       1                0.5
1    A       0                0.5
2    B       1                0.5
3    B       0                0.5


### Explanation
Using the full dataset target to encode categories leaks validation and test outcomes. For training rows, out-of-fold target encoding is safer than using the row’s own target. Validation and test rows must use mappings learned from training data only. Smoothing and unknown-category handling are also important.

## Detecting Leakage

### Definition
Detecting Leakage is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply detecting leakage while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, detecting leakage can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [17]:
suspect=pd.DataFrame({'Age':customers['Age'],'Target_Copy':customers['Repeat_Purchase'],'Target':customers['Repeat_Purchase']});print(suspect.corr(numeric_only=True)['Target'].round(3))

Age            0.048
Target_Copy    1.000
Target         1.000
Name: Target, dtype: float64


### Explanation
An unusually perfect relationship can be a warning, not proof, of leakage. Check feature names, timestamps, collection processes, and target definitions. Compare random and time-based evaluation when appropriate. Audit engineered aggregations and encoders.

## Preventing Leakage

### Definition
Preventing Leakage is a useful concept in feature engineering. It describes a specific way to represent, inspect, or transform information. Its purpose is to make patterns easier to understand without changing what the records mean. The right choice depends on the dataset and the prediction goal.

### Business Example
In a retail business, customer ages, purchase amounts, membership levels, and dates contain different kinds of information. A team may apply preventing leakage while studying repeat purchases. The result can help summarize customer behavior or prepare a reliable prediction system. Business context determines whether the resulting feature is meaningful.

### AI/ML Use
In machine learning, preventing leakage can affect what a model learns from the training data. It should be applied only when the information would be available at prediction time. Any learned parameters must be fitted using training data rather than test data. The usefulness of the result should be checked on held-out observations.

### Python Example

In [19]:
features=customers[['Age','Annual_Income','City']];train,test=train_test_split(features,test_size=.25,random_state=42);print('Fit on train:',train.shape,'Transform test:',test.shape)

Fit on train: (90, 3) Transform test: (30, 3)


### Explanation
This example demonstrates preventing leakage with the customer shopping dataset. Run the cell to inspect its result. Notice which columns are used and what changes in the output. In a real project, compare the result with the original data and check that no future or target information has leaked.